In [18]:


import pandas as pd
import numpy as np
from datetime import datetime



# PART 1: Build a price estimator from the historical monthly data

def build_price_estimator(csv_path="Nat_Gas.csv"):

    df = pd.read_csv(csv_path)
    df["Dates"] = pd.to_datetime(df["Dates"])
    df = df.sort_values("Dates").reset_index(drop=True)

    # Represent every date as a number (days since the first date) so we
    # can use numpy's fast linear interpolation.
    date_nums = df["Dates"].map(lambda d: d.toordinal()).to_numpy()
    prices = df["Prices"].to_numpy()

    def estimate_price(date_like):
        d = pd.to_datetime(date_like)
        x = d.toordinal()
        # np.interp automatically clips to the first/last known price
        # if x falls outside the observed range (flat extrapolation).
        return float(np.interp(x, date_nums, prices))

    return estimate_price



# PART 2: Price the storage contract

def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    price_estimator,
    injection_rate,
    withdrawal_rate,
    max_storage_volume,
    storage_cost_per_unit_per_month,
    injection_withdrawal_cost_per_unit=0.0,
    injection_volume=None,
    withdrawal_volume=None,
    verbose=False,
):

    # 1. Normalise inputs into a single chronological list of events 
    injection_dates = [pd.to_datetime(d) for d in injection_dates]
    withdrawal_dates = [pd.to_datetime(d) for d in withdrawal_dates]

    def _as_list(vol, default, n):
        if vol is None:
            return [default] * n
        if isinstance(vol, (int, float)):
            return [min(vol, default)] * n
        return list(vol)

    inj_vols = _as_list(injection_volume, injection_rate, len(injection_dates))
    wd_vols = _as_list(withdrawal_volume, withdrawal_rate, len(withdrawal_dates))

    events = []
    for d, v in zip(injection_dates, inj_vols):
        events.append({"date": d, "type": "inject", "volume": min(v, injection_rate)})
    for d, v in zip(withdrawal_dates, wd_vols):
        events.append({"date": d, "type": "withdraw", "volume": min(v, withdrawal_rate)})

    events.sort(key=lambda e: e["date"])

    #  2. Walk through time, simulating storage level & cash flows 
    current_volume = 0.0
    total_injection_cost = 0.0
    total_withdrawal_revenue = 0.0
    total_storage_cost = 0.0
    total_handling_cost = 0.0

    last_date = events[0]["date"] if events else None

    for e in events:
        #  a) Charge storage rent for the time gas sat in the tank since
        #        the last event (proportional to volume held & days passed)
        days_elapsed = (e["date"] - last_date).days
        storage_cost = (
            current_volume
            * storage_cost_per_unit_per_month
            * (days_elapsed / 30.0)
        )
        total_storage_cost += storage_cost
        last_date = e["date"]

        #  b) Apply the injection or withdrawal, respecting physical limits
        price = price_estimator(e["date"])
        handling_fee = e["volume"] * injection_withdrawal_cost_per_unit

        if e["type"] == "inject":
            room_left = max_storage_volume - current_volume
            actual_volume = min(e["volume"], room_left)  # can't overfill
            cost = actual_volume * price + handling_fee
            current_volume += actual_volume
            total_injection_cost += cost
            e.update(actual_volume=actual_volume, price=price, cash_flow=-cost)
        else:  # withdraw
            actual_volume = min(e["volume"], current_volume)  # can't withdraw more than stored
            revenue = actual_volume * price - handling_fee
            current_volume -= actual_volume
            total_withdrawal_revenue += revenue
            e.update(actual_volume=actual_volume, price=price, cash_flow=+revenue)

        total_handling_cost += handling_fee

    contract_value = (
        total_withdrawal_revenue
        - total_injection_cost
        - total_storage_cost
    )
    # (handling cost is already netted into injection_cost/withdrawal_revenue
    #  above, so it's not subtracted again -- shown separately just for info)

    if verbose:
        print(f"{'Date':<12}{'Type':<10}{'Volume':>10}{'Price':>10}{'CashFlow':>12}")
        for e in events:
            print(
                f"{e['date'].strftime('%Y-%m-%d'):<12}"
                f"{e['type']:<10}"
                f"{e['actual_volume']:>10.1f}"
                f"{e['price']:>10.2f}"
                f"{e['cash_flow']:>12.2f}"
            )
        print("-" * 54)
        print(f"Total injection cost      : ${total_injection_cost:,.2f}")
        print(f"Total withdrawal revenue  : ${total_withdrawal_revenue:,.2f}")
        print(f"Total storage cost        : ${total_storage_cost:,.2f}")
        print(f"Total handling fees (incl): ${total_handling_cost:,.2f}")
        print(f"CONTRACT VALUE            : ${contract_value:,.2f}")

    return {
        "contract_value": contract_value,
        "total_injection_cost": total_injection_cost,
        "total_withdrawal_revenue": total_withdrawal_revenue,
        "total_storage_cost": total_storage_cost,
        "total_handling_cost": total_handling_cost,
        "events": events,
    }



# TEST CASES

if __name__ == "__main__":

    estimate_price = build_price_estimator("Nat_Gas.csv")

    print("=" * 60)
    print("TEST 1: Simple single injection / single withdrawal")
    print("(buy low in summer, sell high in winter -- classic trade)")
    print("=" * 60)
    result1 = price_storage_contract(
        injection_dates=["2021-06-30"],
        withdrawal_dates=["2021-12-31"],
        price_estimator=estimate_price,
        injection_rate=1_000_000,       # can inject up to 1,000,000 units/day
        withdrawal_rate=1_000_000,      # can withdraw up to 1,000,000 units/day
        max_storage_volume=1_000_000,
        storage_cost_per_unit_per_month=0.05,
        injection_withdrawal_cost_per_unit=0.01,
        verbose=True,
    )

    print("\n" + "=" * 60)
    print("TEST 2: Multiple injections and withdrawals, capped storage")
    print("(client stores gas across several months in smaller batches)")
    print("=" * 60)
    result2 = price_storage_contract(
        injection_dates=["2021-04-30", "2021-05-31", "2021-06-30"],
        withdrawal_dates=["2021-11-30", "2021-12-31"],
        price_estimator=estimate_price,
        injection_rate=300_000,
        withdrawal_rate=400_000,
        max_storage_volume=800_000,     # tighter cap -> tests the volume constraint
        storage_cost_per_unit_per_month=0.03,
        injection_withdrawal_cost_per_unit=0.005,
        verbose=True,
    )

    print("\n" + "=" * 60)
    print("TEST 3: A bad trade -- buying high, selling low")
    print("(sanity check: contract value should come out negative)")
    print("=" * 60)
    result3 = price_storage_contract(
        injection_dates=["2022-02-28"],   # price was high (~11.80)
        withdrawal_dates=["2022-06-30"],  # price had dropped (~10.40)
        price_estimator=estimate_price,
        injection_rate=500_000,
        withdrawal_rate=500_000,
        max_storage_volume=500_000,
        storage_cost_per_unit_per_month=0.02,
        verbose=True,
    )

    print("\n" + "=" * 60)
    print("TEST 4: Storage cap forces gas to be 'left behind'")
    print("(injecting more than the tank can hold on top of existing stock)")
    print("=" * 60)
    result4 = price_storage_contract(
        injection_dates=["2023-01-31", "2023-02-28"],
        withdrawal_dates=["2023-06-30"],
        price_estimator=estimate_price,
        injection_rate=600_000,
        withdrawal_rate=1_000_000,
        max_storage_volume=800_000,      # less than 2 x injection_rate -> triggers cap
        storage_cost_per_unit_per_month=0.04,
        verbose=True,
    )

TEST 1: Simple single injection / single withdrawal
(buy low in summer, sell high in winter -- classic trade)
Date        Type          Volume     Price    CashFlow
2021-06-30  inject     1000000.0     10.00-10010000.00
2021-12-31  withdraw   1000000.0     11.40 11390000.00
------------------------------------------------------
Total injection cost      : $10,010,000.00
Total withdrawal revenue  : $11,390,000.00
Total storage cost        : $306,666.67
Total handling fees (incl): $20,000.00
CONTRACT VALUE            : $1,073,333.33

TEST 2: Multiple injections and withdrawals, capped storage
(client stores gas across several months in smaller batches)
Date        Type          Volume     Price    CashFlow
2021-04-30  inject      300000.0     10.40 -3121500.00
2021-05-31  inject      300000.0      9.84 -2953500.00
2021-06-30  inject      200000.0     10.00 -2001500.00
2021-11-30  withdraw    400000.0     11.20  4478000.00
2021-12-31  withdraw    400000.0     11.40  4558000.00
-----------